# Reproduction attempt — *Unveiling the Compositional Ability Gap in Vision-Language Reasoning Model* (Li et al., NeurIPS 2025)

## Feasibility: **it does not fit**

The paper trains Qwen2.5-VL-3B/7B with SFT, GRPO RL, SFT-init RL and RL-Ground on **4× NVIDIA H100**, generating **8 completions per prompt** (Sec 4.2). GRPO on a 3B VLM needs policy weights + reference weights + optimizer state + 8 concurrent rollouts; that does not fit in 16 GB per T4, and a LoRA variant would not finish 4K+4K samples in a 12-hour session.

The mounted data is also **not the paper's data**. The paper evaluates on ComPABench (synthetic shape/grid tasks the authors generate, Sec 4.1) and zero-shot Geometry3K (Sec 4.4). GQA-CoT appears nowhere in the paper.

This notebook is therefore an **adaptation**, not a reproduction: inference-only, single backbone, on GQA-CoT val. It isolates the one component of RL-Ground that is testable without training — the **caption-before-think** structure (Sec 4.3.2, Fig 6) — plus a grounding probe in the spirit of Sec 4.4.

## Deviation table

| Item | Paper's value | My value | Effect on results |
|---|---|---|---|
| Compute | 4× H100 (Sec 4.2) | 1× T4 16 GB used (2 available; 3B fp16 fits on one) | No training possible; inference only |
| Training | SFT (Eq 1), GRPO RL (Eq 2), SFT-init RL, RL-Ground; 1 epoch, lr 1e-6, 8 completions/prompt (Sec 4.2) | **None.** Zero-shot base model | The paper's central SFT-vs-RL-vs-RL-Ground comparison is **not** reproduced. Only the prompt-structure component of RL-Ground is isolated |
| Progress reward (RL-Ground, Fig 6) | Dense reward on intermediate subgoals | Not implemented (requires RL) | Half of RL-Ground is absent; caption-before-think is tested alone |
| Backbone | Qwen2.5-VL-3B-Instruct **and** 7B-Instruct (Sec 4.2) | 3B-Instruct only | No scale trend (paper reports scale-dependent effects, Sec 4.3.3) |
| Dataset | ComPABench synthetic; Geometry3K zero-shot (Sec 4.1, 4.4) | GQA-CoT val | Different task family. Numbers are **not** comparable to Table 2 / Figs 3–7 |
| Eval size | 500 samples per task type (Sec 4.1) | `EVAL_LIMIT` records sampled from 9,855 (default 1000) | Sampling error; a subset score is not a full-split score |
| Inference repeats | 3 runs averaged (Checklist item 7) | 1 run, greedy | No run-to-run variance estimate |
| Precision | NOT SPECIFIED | fp16 (T4 has no bf16) | Possible small numeric drift |
| Grounding eval | Shape-Area / Grid-Position grounding accuracy (Sec 4.4, Table 2) | Predicted-box IoU vs GQA `bboxs` | Different grounding definition; adaptation only |

**No results appear in this notebook.** Every number in `results.csv` is computed at runtime from real inference on real images.

# Part 1 — Paper analysis

## 1. Goal and proposed architecture

The paper asks whether VLMs post-trained with RL can *compose* skills learned separately, across modalities and across tasks (RQ1–RQ3, Sec 1). It introduces **ComPABench**, a synthetic diagnostic benchmark of geometric-reasoning and spatial-reasoning tasks in paired pure-text / multimodal forms plus compositional and OOD variants (Sec 4.1, Fig 2). No new network architecture is proposed: the backbone is an off-the-shelf Qwen2.5-VL (Sec 4.2), and the contribution lies in the post-training objective. The proposed fix, **RL-Ground**, is GRPO training with two additions: a `<caption>` block forcing visual-to-text description before reasoning, and a fine-grained progress reward over intermediate vision-grounded subgoals (Sec 4.3.2, Fig 6). Reward is the sum of format, progress and accuracy terms (Fig 6).

## 2. Module table

| Module | Function | Input shape | Output shape | Section |
|---|---|---|---|---|
| Vision encoder (Qwen2.5-VL ViT) | Encode image into visual tokens | `(B, 3, H, W)`, dynamic res | `(B, N_v, d)` | Sec 4.2 (backbone only) |
| LLM decoder (Qwen2.5-VL) | Autoregressive generation over interleaved tokens | `(B, N_v + N_t)` | logits `(B, T, V)` | Sec 4.2 |
| `<caption>` block (RL-Ground) | Force visual→text alignment before reasoning | generated prefix | text span | Sec 4.3.2, Fig 6 |
| `<think>` block | Reasoning trace | generated span | text span | Sec 3.1 |
| `<answer>` block | Final response | generated span | text span | Sec 3.1 |
| Format reward | Checks tag structure | generated string | scalar `+1` | Fig 6, Sec 3.2 |
| Progress reward | Subgoal correctness (e.g. per-shape area) | intermediate steps | scalar in `[0,1]` | Sec 4.3.2, Fig 6 |
| Accuracy reward | Final answer match | `<answer>` vs GT | scalar | Sec 3.2 |
| GRPO objective | Group-relative advantage, KL to `π_ref` | G rollouts | scalar loss | Eq 2, Sec 3.2 |
| SFT objective | Token NLL | `(x, y)` | scalar loss | Eq 1, Sec 3.1 |

## 3. Data flow

1. Task instance rendered as pure-text **or** image + question (Sec 4.1, Fig 2).
2. Image → ViT → visual tokens; question → text tokens (Sec 4.2).
3. Decoder generates `<caption>` (RL-Ground only) → `<think>` → `<answer>` (Sec 3.1, Fig 6).
4. Training: SFT = NLL on the full target sequence (Eq 1); RL = sample G=8 rollouts, score with format + accuracy (+ progress for RL-Ground), compute group-relative advantage, GRPO update with β=0 (Eq 2, Sec 4.2).
5. Eval: parse `<answer>`, compare to ground truth, report accuracy (%) on individual, compositional and OOD splits (Sec 4.3, Figs 3/5/7, Table 2).

## 4. Reproduction table

| Item | Paper value | Reference |
|---|---|---|
| Dataset | ComPABench (self-generated); PT-GR, PT-SR, PT-Comp, MM-GR, MM-SR, MM-Comp + OOD variants | Sec 4.1, Table 1, Fig 2 |
| External eval | Geometry3K, zero-shot | Sec 4.4 |
| Split | 4K train samples per data type; 500 eval per type | Sec 4.1 |
| Preprocessing | NOT SPECIFIED (image resolution, tokenization limits not given) | — |
| Feature selection | Not applicable — no explicit feature-selection stage in the paper | — |
| Model per stage | Qwen2.5-VL-3B-Instruct, Qwen2.5-VL-7B-Instruct | Sec 4.2 |
| Loss (SFT) | Token-level NLL | Eq 1, Sec 3.1 |
| Loss (RL) | GRPO with KL term; β set to 0 | Eq 2, Sec 3.2; Sec 4.2 |
| Reward terms | Answer correctness + format adherence; RL-Ground adds progress reward | Sec 3.2; Sec 4.3.2, Fig 6 |
| Optimizer | NOT SPECIFIED | — |
| Learning rate / schedule | 1e-6; schedule NOT SPECIFIED | Sec 4.2 |
| Batch size | Per-device batch size 1; 8 completions per prompt in RL | Sec 4.2 |
| Epochs | 1 | Sec 4.2 |
| Regularization | KL scale set to 0; weight decay / dropout NOT SPECIFIED | Sec 4.2 |
| Max generation length | NOT SPECIFIED | — |
| Inference decoding | NOT SPECIFIED | — |
| Metric | Accuracy (%); string-matching rule NOT SPECIFIED | Figs 3–7, Table 2 |
| Error bars | None reported; inference averaged over 3 runs | Checklist item 7 |
| Ablations (main text) | SFT vs RL vs SFT-init RL vs RL-Ground; 3B vs 7B; pure-text-init vs direct MM training | Sec 4.3.1–4.3.3 |
| Codebase | GRPO implementation based on R1-V | Sec 4.2 |

## 5. Assumptions table

Every row is a choice **I** made because the paper is silent. None of these are paper claims.

| # | Assumption | Value | Why needed |
|---|---|---|---|
| A1 | Inference dtype | fp16 | T4 lacks bf16; paper's precision NOT SPECIFIED |
| A2 | Decoding | greedy, `do_sample=False` | Paper does not specify decoding |
| A3 | Max new tokens | 32 / 256 / 384 / 96 per arm | Paper does not specify generation length |
| A4 | Image token budget | `max_pixels = 768*28*28` | Memory bound on 16 GB; paper does not specify |
| A5 | Answer normalization | NFKC, lowercase, strip punctuation, drop leading articles, collapse whitespace | Paper's match rule NOT SPECIFIED |
| A6 | Parse fallback when `<answer>` missing | use whole decoded string, flag `format_ok=False` | Paper does not describe parse failures |
| A7 | IoU threshold for grounding accuracy | 0.5 | Paper's grounding metric is a different task |
| A8 | Full-frame box threshold | box area ≥ 0.5 × image area | User requirement; paper has no such notion |
| A9 | Predicted-box coordinate space | Qwen2.5-VL absolute coords in post-`smart_resize` space; rescaled by `orig/resized` | Model convention, not paper |
| A10 | Positive class for FPR/FNR | `"yes"` on the yes/no subset | Paper reports no binary metrics |
| A11 | Averaging for Prec/Rec/F1 | macro over the observed answer-string label set | Paper reports accuracy only |
| A12 | Eval subset | 1000 records sampled with seed 42 | 12-hour session limit |
| A13 | Inference batch size | 4 | Throughput/memory; paper's inference batching NOT SPECIFIED |
| A14 | Seed | 42 | Paper does not report seeds |

# Part 2 - Notebook

Sections in order: CONFIG, LOAD DATA, PREPROCESSING, MODEL, TRAIN, EVALUATE, SAVE RESULTS.
There is no FEATURE SELECTION section: the paper has no feature-selection stage.

## Cell 0 - environment

In [ ]:
# ============================================================================
# Cell 0: environment. Pinned installs.
# ============================================================================
!pip install -q --no-warn-conflicts \
    "transformers==4.51.3" \
    "accelerate==1.6.0" \
    "qwen-vl-utils==0.0.11" \
    "scikit-learn==1.5.2"

import os, sys, json, re, time, random, unicodedata, string
import numpy as np
import pandas as pd
import torch
from PIL import Image

import transformers
print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda avail  :", torch.cuda.is_available(), "| n_gpu:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")

SEED = 42  # ASSUMPTION: paper does not specify a seed; using 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# NONDETERMINISM THAT REMAINS: cuBLAS/cuDNN kernel selection on T4, and
# floating-point reduction order that depends on how records fall into batches
# (padding length varies with batch composition). Greedy decoding removes
# sampling noise but not these. torch.use_deterministic_algorithms is NOT set
# because several Qwen2.5-VL kernels have no deterministic implementation.

## CONFIG + mount preflight

Stops on a wrong path and prints what is actually mounted. Never generates substitute data.

In [ ]:
# ============================================================================
# CONFIG  - every path and hyperparameter lives here.
# ============================================================================
CONFIG = {
    # ---- inputs (both mount paths) ----
    "ANN_PATH": "/kaggle/input/notebooks/khoangoo/test-dataset-visual-cot/visual-cot/"
                "cot_with_detailed_reasoning_steps/gqa_cot_val.jsonl",
    "IMAGE_ROOT": "/kaggle/input/datasets/lyte69/gqa-images/images",
    "OUT_DIR": "/kaggle/working",

    # ---- expectations stated by the user, checked not assumed ----
    "EXPECTED_RECORDS": 9855,
    "EXPECTED_UNIQUE_IMAGES": 5422,

    # ---- model (paper Sec 4.2 backbone) ----
    "MODEL_ID": "Qwen/Qwen2.5-VL-3B-Instruct",
    # ASSUMPTION: paper does not specify inference dtype; T4 has no bf16 -> fp16
    "DTYPE": torch.float16,
    "ATTN_IMPL": "sdpa",          # ASSUMPTION: flash-attn2 unsupported on T4 (sm_75)
    # ASSUMPTION: paper does not specify image resolution limits; bounding for 16 GB
    "MIN_PIXELS": 256 * 28 * 28,
    "MAX_PIXELS": 768 * 28 * 28,

    # ---- training (paper Sec 4.2) - NOT RUN, see deviation table ----
    "TRAIN_ENABLED": False,
    "PAPER_LR": 1e-6,             # Sec 4.2
    "PAPER_EPOCHS": 1,            # Sec 4.2
    "PAPER_PER_DEVICE_BS": 1,     # Sec 4.2
    "PAPER_NUM_GENERATIONS": 8,   # Sec 4.2
    "PAPER_KL_BETA": 0.0,         # Sec 4.2

    # ---- evaluation ----
    # ASSUMPTION: paper evaluates 500/task on its own benchmark (Sec 4.1);
    # here we subsample GQA-CoT val to fit the 12h session. Set None for all records.
    "EVAL_LIMIT": 1000,
    "BATCH_SIZE": 4,              # ASSUMPTION: paper does not specify inference batching
    "ARMS": ["direct", "think", "caption_think", "ground"],
    # ASSUMPTION: paper does not specify generation length
    "MAX_NEW_TOKENS": {"direct": 32, "think": 256, "caption_think": 384, "ground": 96},
    # ASSUMPTION: paper does not specify decoding; greedy
    "DO_SAMPLE": False,
    "N_INFERENCE_REPEATS": 1,     # paper checklist item 7 averages 3; see deviation table

    # ---- metric thresholds (NOT from the paper) ----
    "IOU_THRESHOLD": 0.5,            # ASSUMPTION: paper's grounding metric differs; 0.5 default
    "FULL_FRAME_AREA_FRAC": 0.5,     # ASSUMPTION: user requirement; not a paper notion
    "BINARY_POSITIVE_CLASS": "yes",  # ASSUMPTION: paper reports no binary metrics

    # ---- bookkeeping for the results table ----
    "SOURCE": "Li et al., NeurIPS 2025 - Unveiling the Compositional Ability Gap in VLM Reasoning",
    "SPLIT": "gqa_cot_val (visual-cot)",
    "FIDELITY": "adaptation",
    "SEED": SEED,
}
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)

# ---------------------------------------------------------------------------
# PREFLIGHT: mounts must exist. If they do not, print what IS mounted and stop.
# Missing data means a wrong path, never a reason to synthesise anything.
# ---------------------------------------------------------------------------
def _dump_mounts():
    print("\n--- what is actually mounted under /kaggle/input ---")
    if not os.path.isdir("/kaggle/input"):
        print("  /kaggle/input does not exist")
        return
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - 2
        if depth > 4:
            dirs[:] = []
            continue
        print("  " * depth + os.path.basename(root) + f"/   [{len(files)} files]")

print("ANN_PATH   :", CONFIG["ANN_PATH"])
print("IMAGE_ROOT :", CONFIG["IMAGE_ROOT"])

if not os.path.isfile(CONFIG["ANN_PATH"]):
    _dump_mounts()
    raise FileNotFoundError(f"Annotation file not found: {CONFIG['ANN_PATH']}")
if not os.path.isdir(CONFIG["IMAGE_ROOT"]):
    _dump_mounts()
    raise FileNotFoundError(f"IMAGE_ROOT not found: {CONFIG['IMAGE_ROOT']}")

_t0 = time.time()
IMAGE_FILES = os.listdir(CONFIG["IMAGE_ROOT"])
print(f"files in IMAGE_ROOT: {len(IMAGE_FILES)}  (listed in {time.time()-_t0:.1f}s)")
if len(IMAGE_FILES) == 0:
    _dump_mounts()
    raise RuntimeError("IMAGE_ROOT is empty.")
print("sample filenames    :", IMAGE_FILES[:8])

## LOAD DATA

Resolves every unique image and verifies on-disk size against the annotated `width`/`height`, because `bboxs` are pixel coordinates.

In [ ]:
# ============================================================================
# LOAD DATA
# ============================================================================
raw_records = []
with open(CONFIG["ANN_PATH"], "r") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_records.append(json.loads(line))

print(f"records loaded      : {len(raw_records)}  (expected {CONFIG['EXPECTED_RECORDS']})")
if len(raw_records) != CONFIG["EXPECTED_RECORDS"]:
    print(f"WARNING: record count differs from the stated {CONFIG['EXPECTED_RECORDS']}. "
          "Downstream n_eval reflects the file that is actually mounted.")

required_fields = ["question", "answer", "image", "width", "height", "bboxs"]
missing_field_rows = [i for i, r in enumerate(raw_records)
                      if any(k not in r for k in required_fields)]
if missing_field_rows:
    print("first rows missing required fields:", missing_field_rows[:20])
    raise KeyError(f"{len(missing_field_rows)} records lack one of {required_fields}")

unique_images = sorted({r["image"] for r in raw_records})
print(f"unique images in ann: {len(unique_images)}  (expected {CONFIG['EXPECTED_UNIQUE_IMAGES']})")
print(f"records per image   : {len(raw_records)/max(len(unique_images),1):.2f}")

# ---- 1) do all unique images resolve on disk? ----
image_set = set(IMAGE_FILES)
missing = [fn for fn in unique_images if fn not in image_set]
print(f"unique images resolving on disk: {len(unique_images)-len(missing)}/{len(unique_images)}")
if missing:
    print("first 20 missing filenames:")
    for fn in missing[:20]:
        print("   ", fn)
    raise FileNotFoundError(
        f"{len(missing)} unique images do not resolve under IMAGE_ROOT. "
        "Fix the path. Substitute images are never generated."
    )

# ---- 2) do the on-disk sizes match the annotated width/height? ----
# bboxs are pixel coordinates: a re-encoded / resized image puts the box in the
# wrong place, so those records must be dropped, not corrected.
_t0 = time.time()
real_size = {}
unreadable = []
for k, fn in enumerate(unique_images):
    path = os.path.join(CONFIG["IMAGE_ROOT"], fn)
    try:
        with Image.open(path) as im:      # header read only, no decode
            real_size[fn] = im.size       # (W, H)
    except (OSError, ValueError) as e:
        unreadable.append((fn, repr(e)))
    if (k + 1) % 1000 == 0:
        print(f"  probed {k+1}/{len(unique_images)} ({time.time()-_t0:.0f}s)")
print(f"probed {len(unique_images)} image headers in {time.time()-_t0:.1f}s")

if unreadable:
    print("first 20 unreadable images:")
    for fn, e in unreadable[:20]:
        print("   ", fn, e)
    raise OSError(f"{len(unreadable)} images could not be opened.")

size_ok_records, size_bad = [], []
for r in raw_records:
    w_disk, h_disk = real_size[r["image"]]
    if int(r["width"]) == int(w_disk) and int(r["height"]) == int(h_disk):
        size_ok_records.append(r)
    else:
        size_bad.append((r["image"], (r["width"], r["height"]), (w_disk, h_disk)))

print(f"\nsize-mismatch records dropped: {len(size_bad)} / {len(raw_records)} "
      f"({100*len(size_bad)/max(len(raw_records),1):.2f}%)")
if size_bad:
    print("  first 10 mismatches (file, annotated WxH, on-disk WxH):")
    for row in size_bad[:10]:
        print("   ", row)
    frac = len(size_bad) / len(raw_records)
    if frac > 0.10:
        print("  NOTE: >10% mismatch means the image set was re-encoded/resized. "
              "The pixel bboxs cannot be trusted for the dropped records; grounding "
              "numbers below are computed only on the size-verified remainder.")
print(f"records surviving size verification: {len(size_ok_records)}")
if len(size_ok_records) == 0:
    raise RuntimeError("No records survived size verification. Stopping.")

## PREPROCESSING

Clamps boxes to the frame, flags full-frame boxes, samples the eval subset, defines the four prompt arms.

In [ ]:
# ============================================================================
# PREPROCESSING
# ============================================================================
def clamp_boxes(boxes, W, H):
    """Clamp [x1,y1,x2,y2] to [0,W]x[0,H]. Returns (clamped_boxes, n_changed)."""
    out, changed = [], 0
    for b in boxes:
        x1, y1, x2, y2 = [float(v) for v in b[:4]]
        cx1, cy1 = max(0.0, min(x1, W)), max(0.0, min(y1, H))
        cx2, cy2 = max(0.0, min(x2, W)), max(0.0, min(y2, H))
        if (cx1, cy1, cx2, cy2) != (x1, y1, x2, y2):
            changed += 1
        if cx2 < cx1: cx1, cx2 = cx2, cx1
        if cy2 < cy1: cy1, cy2 = cy2, cy1
        out.append([cx1, cy1, cx2, cy2])
    return out, changed


def enclosing_box(boxes):
    xs1 = min(b[0] for b in boxes); ys1 = min(b[1] for b in boxes)
    xs2 = max(b[2] for b in boxes); ys2 = max(b[3] for b in boxes)
    return [xs1, ys1, xs2, ys2]


clean, n_boxes_clamped, n_recs_clamped, n_no_box = [], 0, 0, 0
for r in size_ok_records:
    W, H = int(r["width"]), int(r["height"])
    boxes = r.get("bboxs") or []
    boxes = [b for b in boxes if isinstance(b, (list, tuple)) and len(b) >= 4]
    if not boxes:
        n_no_box += 1
        gt_boxes, enc, full_frame = [], None, False
    else:
        gt_boxes, ch = clamp_boxes(boxes, W, H)
        if ch:
            n_boxes_clamped += ch
            n_recs_clamped += 1
        enc = enclosing_box(gt_boxes)
        area = max(0.0, enc[2] - enc[0]) * max(0.0, enc[3] - enc[1])
        # ASSUMPTION: "covers most of the frame" is not a paper notion;
        # using area >= FULL_FRAME_AREA_FRAC * image area
        full_frame = (W * H > 0) and (area >= CONFIG["FULL_FRAME_AREA_FRAC"] * W * H)
    clean.append({
        "image": r["image"],
        "path": os.path.join(CONFIG["IMAGE_ROOT"], r["image"]),
        "question": r["question"],
        "answer": str(r["answer"]),
        "full_answer": r.get("full_answer", ""),
        "width": W, "height": H,
        "gt_boxes": gt_boxes,
        "gt_enclosing": enc,
        "is_full_frame": bool(full_frame),
        "has_box": len(gt_boxes) > 0,
    })

print(f"boxes clamped       : {n_boxes_clamped} (in {n_recs_clamped} records)")
print(f"records with no box : {n_no_box}")
n_ff = sum(x["is_full_frame"] for x in clean)
print(f"full-frame records  : {n_ff} / {len(clean)} "
      f"({100*n_ff/max(len(clean),1):.1f}%)  -> grounding reported with AND without these")

# ---- eval subset (deviation row: subset scores are not full-split scores) ----
rng = random.Random(CONFIG["SEED"])
eval_records = list(clean)
if CONFIG["EVAL_LIMIT"] is not None and CONFIG["EVAL_LIMIT"] < len(eval_records):
    eval_records = rng.sample(eval_records, CONFIG["EVAL_LIMIT"])
    print(f"\nDEVIATION: scoring {len(eval_records)} of {len(clean)} verified records "
          f"(file has {len(raw_records)}). Recorded as n_eval.")
else:
    print(f"\nscoring all {len(eval_records)} verified records")

# ---- prompt templates ----
# 'think'          -> paper Sec 3.1: output has a <think> block and an <answer> block.
# 'caption_think'  -> RL-Ground format, Sec 4.3.2 / Fig 6: <caption> then <think> then <answer>.
# 'direct'         -> answer-only control (no reasoning trace).
# 'ground'         -> ADAPTATION. The paper's grounding probes are Shape-Area and
#                     Grid-Position grounding (Sec 4.4); GQA has pixel bboxs instead.
PROMPTS = {
    "direct":
        "{q}\nAnswer with a single word or short phrase inside <answer></answer> tags.",
    "think":
        "{q}\nFirst reason step by step inside <think></think> tags. "
        "Then give the final short answer (a single word or phrase) inside <answer></answer> tags.",
    "caption_think":
        "{q}\nFirst describe the visual content of the image in natural language inside "
        "<caption></caption> tags. Then reason step by step inside <think></think> tags. "
        "Then give the final short answer (a single word or phrase) inside <answer></answer> tags.",
    "ground":
        "{q}\nFirst output the bounding box of the image region needed to answer, as "
        "<box>[x1,y1,x2,y2]</box> in pixel coordinates of the image you were given. "
        "Then give the final short answer (a single word or phrase) inside <answer></answer> tags.",
}
print("\narms:", CONFIG["ARMS"])

## MODEL

Qwen2.5-VL-3B-Instruct (paper Sec 4.2 backbone), fp16, zero-shot. Runs on 1 GPU or CPU without edits.

In [ ]:
# ============================================================================
# MODEL - Qwen2.5-VL-3B-Instruct, the paper's smaller backbone (Sec 4.2).
# No architectural change. Zero-shot: no SFT/RL weights exist for this setup.
# ============================================================================
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU detected. This will be extremely slow but is not blocked.")
# Both T4s are visible, but a 3B fp16 model needs only one (~6.2 GB of weights),
# so no model sharding is used, per the instruction to use two GPUs only if needed.

processor = AutoProcessor.from_pretrained(
    CONFIG["MODEL_ID"],
    min_pixels=CONFIG["MIN_PIXELS"],
    max_pixels=CONFIG["MAX_PIXELS"],
)
processor.tokenizer.padding_side = "left"   # required for batched generation

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CONFIG["MODEL_ID"],
    torch_dtype=CONFIG["DTYPE"] if DEVICE != "cpu" else torch.float32,
    attn_implementation=CONFIG["ATTN_IMPL"],
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()

N_PARAMS = sum(p.numel() for p in model.parameters())
print(f"model            : {CONFIG['MODEL_ID']}")
print(f"parameters       : {N_PARAMS:,}")
if DEVICE != "cpu":
    torch.cuda.reset_peak_memory_stats()
    print(f"weights on GPU   : {torch.cuda.memory_allocated()/1e9:.2f} GB")


def build_messages(rec, arm):
    return [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": PROMPTS[arm].format(q=rec["question"])},
    ]}]


@torch.inference_mode()
def generate_batch(batch, arm):
    """Returns (list_of_decoded_strings, list_of_(resized_w, resized_h))."""
    texts = [processor.apply_chat_template(build_messages(r, arm),
                                           tokenize=False, add_generation_prompt=True)
             for r in batch]
    images = []
    for r in batch:
        with Image.open(r["path"]) as im:
            images.append(im.convert("RGB"))
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(DEVICE)

    # Qwen2.5-VL emits absolute coordinates in the post-smart_resize input space.
    # image_grid_thw is in 14-px patch units, so resized side = grid * 14.
    grid = inputs["image_grid_thw"].detach().cpu().numpy()       # (B, 3) = (t, h, w)
    resized = [(int(g[2]) * 14, int(g[1]) * 14) for g in grid]   # (W_resized, H_resized)

    out = model.generate(
        **inputs,
        max_new_tokens=CONFIG["MAX_NEW_TOKENS"][arm],
        do_sample=CONFIG["DO_SAMPLE"],
        pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
    )
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    decoded = processor.batch_decode(new_tokens, skip_special_tokens=True)
    return [d.strip() for d in decoded], resized

## TRAIN

Not run. See the deviation table at the top.

In [ ]:
# ============================================================================
# TRAIN
# The paper trains SFT (Eq 1), GRPO RL (Eq 2), SFT-init RL (Sec 3.3) and
# RL-Ground (Sec 4.3.2) on 4x H100 with 8 completions per prompt (Sec 4.2).
# That does not fit on 2x T4 16 GB inside a 12h session -> this run performs
# NO training. Training Steps is therefore 0 and Train Time is 0.0 seconds.
# These are facts about this run, not measurements of the paper's setup.
# ============================================================================
TRAIN_TIME_S = 0.0
TRAINING_STEPS = 0

if CONFIG["TRAIN_ENABLED"]:
    raise NotImplementedError(
        "TRAIN_ENABLED=True but no training procedure is implemented here. "
        "GRPO as specified in Sec 4.2 (per-device batch 1, 8 completions/prompt, "
        "lr 1e-6, 1 epoch, KL beta 0) requires policy + reference weights + 8 "
        "concurrent rollouts of a 3B VLM, which exceeds 16 GB per T4. "
        "Implementing a shrunken variant silently would break comparability, so "
        "this path stops instead of guessing."
    )

print("TRAIN: skipped by design (see deviation table).")
print(f"  paper training config for reference: lr={CONFIG['PAPER_LR']}, "
      f"epochs={CONFIG['PAPER_EPOCHS']}, per_device_bs={CONFIG['PAPER_PER_DEVICE_BS']}, "
      f"G={CONFIG['PAPER_NUM_GENERATIONS']}, kl_beta={CONFIG['PAPER_KL_BETA']}  (Sec 4.2)")
print(f"  TRAIN_TIME_S={TRAIN_TIME_S}  TRAINING_STEPS={TRAINING_STEPS}")

## EVALUATE

Normalized exact match on every eval record. Real inference, no sampled correctness flags.

In [ ]:
# ============================================================================
# EVALUATE
# Primary metric = normalized exact match on the short answer string, matching
# the paper's use of accuracy (%) (Figs 3,5,7; Table 2). The paper does not
# define its string-matching rule -> normalization below is assumption A5.
# ============================================================================
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

_PUNCT = str.maketrans("", "", string.punctuation)
_ARTICLES = {"a", "an", "the"}


def normalize_answer(s):
    """ASSUMPTION A5: NFKC -> lowercase -> strip punctuation -> drop leading
    articles -> collapse whitespace. The paper does not specify a match rule."""
    s = unicodedata.normalize("NFKC", str(s)).lower().strip()
    s = s.translate(_PUNCT)
    toks = [t for t in s.split() if t]
    while toks and toks[0] in _ARTICLES:
        toks = toks[1:]
    return " ".join(toks)


ANS_RE = re.compile(r"<answer>(.*?)</answer>", re.DOTALL | re.IGNORECASE)
BOX_TAG_RE = re.compile(r"<box>\s*\[?([^\]<]+?)\]?\s*</box>", re.DOTALL | re.IGNORECASE)
BOX_ANY_RE = re.compile(r"\[\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)\s*,"
                        r"\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)\s*\]")


def parse_answer(text):
    """Returns (answer_string, format_ok). ASSUMPTION A6: if the <answer> tag is
    absent, fall back to the whole decoded string and flag format_ok=False."""
    m = ANS_RE.search(text)
    if m:
        return m.group(1).strip(), True
    return text.strip(), False


def format_adherence(text, arm):
    """Paper's format reward checks tag structure (Sec 3.2, Fig 6)."""
    ok = bool(ANS_RE.search(text))
    if arm in ("think", "caption_think"):
        ok = ok and ("<think>" in text.lower())
    if arm == "caption_think":
        ok = ok and ("<caption>" in text.lower())
    if arm == "ground":
        ok = ok and bool(BOX_TAG_RE.search(text))
    return ok


def parse_box(text):
    """Returns [x1,y1,x2,y2] in the model's input (resized) coordinate space, or None."""
    m = BOX_TAG_RE.search(text)
    if m:
        parts = [p for p in re.split(r"[,\s]+", m.group(1).strip()) if p]
        try:
            vals = [float(p) for p in parts[:4]]
        except ValueError:
            vals = []
        if len(vals) == 4:
            return vals
    m2 = BOX_ANY_RE.search(text)
    if m2:
        return [float(m2.group(i)) for i in range(1, 5)]
    return None


def rescale_box(box, resized_wh, orig_wh):
    """ASSUMPTION A9: Qwen2.5-VL emits absolute coords in post-smart_resize space."""
    rw, rh = resized_wh
    ow, oh = orig_wh
    if rw <= 0 or rh <= 0:
        return None
    sx, sy = ow / rw, oh / rh
    x1, y1, x2, y2 = box
    return [x1 * sx, y1 * sy, x2 * sx, y2 * sy]


def iou(a, b):
    ax1, ay1, ax2, ay2 = min(a[0], a[2]), min(a[1], a[3]), max(a[0], a[2]), max(a[1], a[3])
    bx1, by1, bx2, by2 = min(b[0], b[2]), min(b[1], b[3]), max(b[0], b[2]), max(b[1], b[3])
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / union if union > 0 else float("nan")


def run_arm(arm, records):
    """Runs real inference on every record and returns per-sample rows."""
    rows, bs = [], CONFIG["BATCH_SIZE"]
    t_start = time.time()
    for s in range(0, len(records), bs):
        batch = records[s:s + bs]
        decoded, resized = generate_batch(batch, arm)
        for rec, text, rwh in zip(batch, decoded, resized):
            pred_ans, fmt_answer_ok = parse_answer(text)
            row = {
                "image": rec["image"], "question": rec["question"],
                "gold": rec["answer"], "pred_raw": text, "pred_answer": pred_ans,
                "gold_norm": normalize_answer(rec["answer"]),
                "pred_norm": normalize_answer(pred_ans),
                "answer_tag_present": fmt_answer_ok,
                "format_ok": format_adherence(text, arm),
                "is_full_frame": rec["is_full_frame"], "has_box": rec["has_box"],
                "pred_box": None, "iou": float("nan"),
            }
            row["correct"] = int(row["pred_norm"] == row["gold_norm"])
            if arm == "ground" and rec["has_box"]:
                pb = parse_box(text)
                if pb is not None:
                    pb = rescale_box(pb, rwh, (rec["width"], rec["height"]))
                if pb is not None:
                    row["pred_box"] = pb
                    ious = [iou(pb, g) for g in rec["gt_boxes"]]
                    ious = [v for v in ious if v == v]
                    row["iou"] = max(ious) if ious else float("nan")
                else:
                    row["iou"] = 0.0   # box requested, none parseable -> no overlap
            rows.append(row)
        done = s + len(batch)
        if (s // bs) % 10 == 0 or done == len(records):
            el = time.time() - t_start
            print(f"  [{arm}] {done}/{len(records)}  {el:.0f}s elapsed  "
                  f"ETA {el/max(done,1)*(len(records)-done):.0f}s", flush=True)
    return rows, time.time() - t_start


def score(rows):
    """Builds the metric dict ONLY from computed values. No defaults, no fills."""
    m = {}
    y_true = [r["gold_norm"] for r in rows]
    y_pred = [r["pred_norm"] for r in rows]
    m["acc"] = float(np.mean([r["correct"] for r in rows]))
    labels = sorted(set(y_true) | set(y_pred))
    # ASSUMPTION A11: macro over the observed answer-string label set; the paper
    # reports accuracy only and specifies no averaging.
    try:
        m["prec_macro"] = float(precision_score(y_true, y_pred, labels=labels,
                                                average="macro", zero_division=0))
        m["recall_macro"] = float(recall_score(y_true, y_pred, labels=labels,
                                               average="macro", zero_division=0))
        m["f1_macro"] = float(f1_score(y_true, y_pred, labels=labels,
                                       average="macro", zero_division=0))
    except ValueError as e:
        print("  macro P/R/F1 failed:", e)
        m["prec_macro"] = m["recall_macro"] = m["f1_macro"] = float("nan")

    # FPR / FNR are defined only on the binary yes/no subset of GQA.
    pos = CONFIG["BINARY_POSITIVE_CLASS"]
    bin_rows = [r for r in rows if r["gold_norm"] in ("yes", "no")]
    if bin_rows:
        bt = [1 if r["gold_norm"] == pos else 0 for r in bin_rows]
        bp = [1 if r["pred_norm"] == pos else 0 for r in bin_rows]
        try:
            tn, fp, fn, tp = confusion_matrix(bt, bp, labels=[0, 1]).ravel()
            m["fpr"] = float(fp / (fp + tn)) if (fp + tn) > 0 else float("nan")
            m["fnr"] = float(fn / (fn + tp)) if (fn + tp) > 0 else float("nan")
        except ValueError as e:
            print("  binary confusion matrix failed:", e)
            m["fpr"] = m["fnr"] = float("nan")
    else:
        m["fpr"] = m["fnr"] = float("nan")
    m["n_binary"] = len(bin_rows)
    m["format_ok_rate"] = float(np.mean([r["format_ok"] for r in rows]))
    m["answer_tag_rate"] = float(np.mean([r["answer_tag_present"] for r in rows]))

    # Grounding: reported twice, with and without full-frame gt boxes.
    g_all = [r["iou"] for r in rows if r["has_box"] and r["iou"] == r["iou"]]
    g_nff = [r["iou"] for r in rows
             if r["has_box"] and not r["is_full_frame"] and r["iou"] == r["iou"]]
    thr = CONFIG["IOU_THRESHOLD"]
    m["iou_mean_all"] = float(np.mean(g_all)) if g_all else float("nan")
    m["iou_acc_all"] = float(np.mean([v >= thr for v in g_all])) if g_all else float("nan")
    m["iou_mean_nofullframe"] = float(np.mean(g_nff)) if g_nff else float("nan")
    m["iou_acc_nofullframe"] = float(np.mean([v >= thr for v in g_nff])) if g_nff else float("nan")
    m["n_grounded_all"] = len(g_all)
    m["n_grounded_nofullframe"] = len(g_nff)
    return m


if DEVICE != "cpu":
    torch.cuda.reset_peak_memory_stats()

ARM_METRICS, ARM_TIMES = {}, {}
for arm in CONFIG["ARMS"]:
    print(f"\n=== arm: {arm} | n={len(eval_records)} | "
          f"max_new_tokens={CONFIG['MAX_NEW_TOKENS'][arm]} ===", flush=True)
    rows, secs = run_arm(arm, eval_records)
    ARM_METRICS[arm] = score(rows)
    ARM_TIMES[arm] = secs
    with open(os.path.join(CONFIG["OUT_DIR"], f"predictions_{arm}.jsonl"), "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"  arm '{arm}' finished in {secs:.0f}s -> predictions_{arm}.jsonl")

PEAK_GPU_GB = (torch.cuda.max_memory_allocated() / 1e9) if DEVICE != "cpu" else float("nan")
print(f"\npeak GPU memory allocated: {PEAK_GPU_GB:.2f} GB")

## SAVE RESULTS

In [ ]:
# ============================================================================
# SAVE RESULTS
# Every cell traces to a variable computed in TRAIN or EVALUATE.
# ROC-AUC / PR-AUC are N/A: the outputs are free-form short strings, there is no
# label-independent score (no predict_proba, no fixed class logits). Deriving a
# score from correctness would return 1.0 by construction and measure nothing.
# Comm Cost is N/A: the paper measures no communication cost.
# ============================================================================
def fmt(v):
    if isinstance(v, float) and v != v:
        return "N/A"
    return v


ARM_LABEL = {
    "direct":        "answer-only control",
    "think":         "<think>+<answer> (paper Sec 3.1 format)",
    "caption_think": "<caption>+<think>+<answer> (RL-Ground format, Sec 4.3.2/Fig 6)",
    "ground":        "box-then-answer (ADAPTATION of Sec 4.4 grounding probe)",
}

rows = []
for arm in CONFIG["ARMS"]:
    m = ARM_METRICS[arm]
    rows.append({
        "Acc": m["acc"],
        "Prec": fmt(m["prec_macro"]),
        "Recall": fmt(m["recall_macro"]),
        "F1": fmt(m["f1_macro"]),
        "ROC-AUC": "N/A",
        "PR-AUC": "N/A",
        "FPR": fmt(m["fpr"]),
        "FNR": fmt(m["fnr"]),
        "Train Time": TRAIN_TIME_S,
        "Params": N_PARAMS,
        "Comm Cost": "N/A",
        "Training Steps": TRAINING_STEPS,
        "n_eval": len(eval_records),
        "source": CONFIG["SOURCE"],
        "split": CONFIG["SPLIT"],
        "model": f"{CONFIG['MODEL_ID']} (3B, fp16, zero-shot) | arm={arm}: {ARM_LABEL[arm]}",
        "inputs": f"{CONFIG['ANN_PATH']} + {CONFIG['IMAGE_ROOT']}",
        "fidelity": CONFIG["FIDELITY"],
        # extras appended after the required columns so the paste still lines up
        "peak_gpu_mem_gb": fmt(PEAK_GPU_GB),
        "eval_time_s": round(ARM_TIMES[arm], 1),
        "format_ok_rate": m["format_ok_rate"],
        "answer_tag_rate": m["answer_tag_rate"],
        "n_binary_yesno": m["n_binary"],
        "iou_mean_all": fmt(m["iou_mean_all"]),
        f"iou_acc@{CONFIG['IOU_THRESHOLD']}_all": fmt(m["iou_acc_all"]),
        "iou_mean_nofullframe": fmt(m["iou_mean_nofullframe"]),
        f"iou_acc@{CONFIG['IOU_THRESHOLD']}_nofullframe": fmt(m["iou_acc_nofullframe"]),
        "n_grounded_all": m["n_grounded_all"],
        "n_grounded_nofullframe": m["n_grounded_nofullframe"],
        "n_records_in_file": len(raw_records),
        "n_dropped_size_mismatch": len(size_bad),
        "n_boxes_clamped": n_boxes_clamped,
        "n_full_frame_records": n_ff,
    })

COL_ORDER = ["Acc", "Prec", "Recall", "F1", "ROC-AUC", "PR-AUC", "FPR", "FNR",
             "Train Time", "Params", "Comm Cost", "Training Steps",
             "n_eval", "source", "split", "model", "inputs", "fidelity"]
results = pd.DataFrame(rows)
results = results[COL_ORDER + [c for c in results.columns if c not in COL_ORDER]]

pd.set_option("display.max_columns", None, "display.width", 250)
display(results)

out_csv = os.path.join(CONFIG["OUT_DIR"], "results.csv")
results.to_csv(out_csv, index=False)
with open(os.path.join(CONFIG["OUT_DIR"], "run_config.json"), "w") as f:
    json.dump({k: (str(v) if not isinstance(v, (int, float, str, bool, list, dict, type(None))) else v)
               for k, v in CONFIG.items()}, f, indent=2)
print("wrote:", out_csv)

print("\nREAD THIS BEFORE COMPARING TO THE PAPER")
print("- fidelity = adaptation. No SFT / RL / SFT-init-RL / RL-Ground training was run;")
print("  GRPO as specified in Sec 4.2 does not fit on 2x T4 (see deviation table).")
print("- The paper's numbers are on ComPABench (Sec 4.1) and Geometry3K (Sec 4.4),")
print("  not GQA. Do not place these rows next to Table 2 or Figs 3/5/7.")
print("- The caption_think arm tests only the <caption> half of RL-Ground (Fig 6);")
print("  the progress reward requires RL and is absent.")
print("- Grounding here is box IoU, a different metric from the paper's grounding tasks.")

## What is missing / where to resume

Not implemented here, each a separate build:

1. **ComPABench generation** (Sec 4.1, Fig 2) — the shape-area / grid-position / compositional / OOD generators. The paper says construction details are in the supplementary, which is not in the PDF, so writing the generator would mean inventing it rather than reproducing it.
2. **GRPO training loop** (Eq 2, Sec 3.2) and **RL-Ground's progress reward** (Fig 6) — these need ComPABench subgoal labels and hardware beyond a T4.
3. **7B backbone and the scale trend** (Sec 4.3.3).

Highest-value next step is item 1: ComPABench generation is pure CPU work, and the pure-text arms (PT-GR, PT-SR, PT-Comp) can be evaluated zero-shot on a single T4. That puts the task family back on the paper's terms, leaving only the training axis as a deviation.